In [ ]:
##The training graphs that are obtained are by running this code, where we consider centers for the population encoding. 
# Taking the activations will also give the same graphs as i tested that in the encoding methods. 
# minimax agent reference code: https://www.geeksforgeeks.org/dsa/finding-optimal-move-in-tic-tac-toe-using-minimax-algorithm-in-game-theory/
# wandb link: https://wandb.ai/kradeero-ohio-university/thesis/runs/kimm99p6
#saved model: tictactoe_dsnn_vs_minimax2.pth

import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import os

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# ---------------------
# Base Model
# ---------------------
class BaseModel:
    """Base class for reinforcement learning models"""
    def __init__(self, discount_factor, epsilon, e_min, e_max):
        self.discount_factor = discount_factor
        self.epsilon = epsilon
        self.e_min = e_min
        self.e_max = e_max

# ---------------------
# Tic-tac-toe Environment with Minimax Agent
# ---------------------
class Tictactoe_v0:
    def __init__(self):
        self.board = [0] * 9
        self.wining_position = [[0, 1, 2], [3, 4, 5], [6, 7, 8],
                                [0, 3, 6], [1, 4, 7], [2, 5, 8],
                                [0, 4, 8], [6, 4, 2]]
        self.current_turn = 1
        self.player_mark = 1

    def reset(self, is_human_first):
        self.board = [0] * 9
        self.current_turn = 1
        self.player_mark = 1 if is_human_first else -1
        # print("\n--- New Episode ---")
        # self.render()
        if not is_human_first:
            self.env_act()
            # self.render()
        return self.board.copy()

    def check_win(self):
        dsqn_symbol = 1 if self.player_mark == 1 else -1  # DQN is X (1) or O (-1)
        opponent_symbol = -dsqn_symbol  # Opponent is O (-1) or X (1)
        for pst in self.wining_position:
            line = [self.board[i] for i in pst]
            if line == [dsqn_symbol, dsqn_symbol, dsqn_symbol]:  # DQN wins
                return 1, True
            elif line == [opponent_symbol, opponent_symbol, opponent_symbol]:  # Opponent wins
                return -1, True
        if 0 not in self.board:
            return 0, True  # Draw
        return 0, False

    def board_to_minimax(self):
        minimax_board = [['_' for _ in range(3)] for _ in range(3)]
        for i in range(9):
            row, col = divmod(i, 3)
            if self.board[i] == 1:
                minimax_board[row][col] = 'x'
            elif self.board[i] == -1:
                minimax_board[row][col] = 'o'
            else:
                minimax_board[row][col] = '_'
        return minimax_board

    def is_moves_left(self, board):
        for i in range(3):
            for j in range(3):
                if board[i][j] == '_':
                    return True
        return False

    def evaluate(self, board, player, opponent):
        for row in range(3):
            if board[row][0] == board[row][1] == board[row][2]:
                if board[row][0] == player:
                    return 10
                elif board[row][0] == opponent:
                    return -10
        for col in range(3):
            if board[0][col] == board[1][col] == board[2][col]:
                if board[0][col] == player:
                    return 10
                elif board[0][col] == opponent:
                    return -10
        if board[0][0] == board[1][1] == board[2][2]:
            if board[0][0] == player:
                return 10
            elif board[0][0] == opponent:
                return -10
        if board[0][2] == board[1][1] == board[2][0]:
            if board[0][2] == player:
                return 10
            elif board[0][2] == opponent:
                return -10
        return 0

    def minimax(self, board, depth, is_max, player, opponent):
        score = self.evaluate(board, player, opponent)
        if score == 10:
            return score - depth
        if score == -10:
            return score + depth
        if not self.is_moves_left(board):
            return 0
        if is_max:
            best = -1000
            for i in range(3):
                for j in range(3):
                    if board[i][j] == '_':
                        board[i][j] = player
                        best = max(best, self.minimax(board, depth + 1, not is_max, player, opponent))
                        board[i][j] = '_'
            return best
        else:
            best = 1000
            for i in range(3):
                for j in range(3):
                    if board[i][j] == '_':
                        board[i][j] = opponent
                        best = min(best, self.minimax(board, depth + 1, not is_max, player, opponent))
                        board[i][j] = '_'
            return best

    def find_best_move(self, board, player, opponent):
        best_val = -1000
        best_moves = []
        for i in range(3):
            for j in range(3):
                if board[i][j] == '_':
                    board[i][j] = player
                    move_val = self.minimax(board, 0, False, player, opponent)
                    board[i][j] = '_'
                    if move_val > best_val:
                        best_moves = [(i, j)]
                        best_val = move_val
                    elif move_val == best_val:
                        best_moves.append((i, j))
        return random.choice(best_moves) if best_moves else (-1, -1)

    def env_act(self):
        minimax_board = self.board_to_minimax()
        player = 'o' if self.current_turn == -1 else 'x'
        opponent = 'x' if player == 'o' else 'o'
        row, col = self.find_best_move(minimax_board, player, opponent)
        if row == -1 and col == -1:
            raise Exception('No valid move found by Minimax')
        action = row * 3 + col
        if self.board[action] != 0:
            raise Exception('Invalid action by Minimax')
        self.board[action] = self.current_turn
        # print(f"Minimax ({'O' if self.current_turn == -1 else 'X'}) plays at position {action}")
        reward, done = self.check_win()
        # print(f"Reward after Minimax move: {reward} (Done: {done})")
        self.current_turn = self.current_turn * -1
        return reward, done

    def step(self, action):
        if self.board[action] != 0:
            raise Exception('Invalid action')
        self.board[action] = self.current_turn
        # print(f"DSQN ({'X' if self.current_turn == 1 else 'O'}) plays at position {action}")
        # self.render()
        reward, done = self.check_win()
        # print(f"Reward after DSQN move: {reward} (Done: {done})")
        self.current_turn = self.current_turn * -1
        if not done:
            reward, done = self.env_act()
            # self.render()
        return self.board.copy(), reward, done, None

    def render(self):
        symbols = {1: 'X', -1: 'O', 0: ' '}
        print("\nCurrent Board:")
        for i in range(3):
            print(f" {symbols[self.board[i*3]]} | {symbols[self.board[i*3+1]]} | {symbols[self.board[i*3+2]]} ")
            if i < 2: print("-----------")
        print()

# ---------------------
# Epsilon Greedy and Experience Replay
# ---------------------
class EpsilonGreedy:
    def __init__(self, epsilon):
        self.epsilon = epsilon

    def perform(self, q_value, action_space: list = None):
        prob = np.random.sample()
        if prob <= self.epsilon:
            if action_space is None:
                return np.random.randint(len(q_value))
            return np.random.choice(action_space)
        else:
            if action_space is None:
                return np.argmax(q_value)
            return max([[q_value[a], a] for a in action_space], key=lambda x: x[0])[1]

    def decay(self, decay_value, lower_bound):
        self.epsilon = max(self.epsilon * decay_value, lower_bound)

class ExperienceReplay:
    def __init__(self, e_max: int):
        if e_max <= 0:
            raise ValueError('Invalid value for memory size')
        self.e_max = e_max
        self.memory = list()
        self.index = 0

    def add_experience(self, sample: list):
        if len(sample) != 5:
            raise Exception('Invalid sample')
        if len(self.memory) < self.e_max:
            self.memory.append(sample)
        else:
            self.memory[self.index] = sample
        self.index = (self.index + 1) % self.e_max

    def sample_experience(self, sample_size: int, cer_mode: bool):
        samples = random.sample(self.memory, sample_size)
        if cer_mode:
            samples[-1] = self.memory[self.index - 1]
        s_batch, a_batch, r_batch, ns_batch, done_batch = map(np.array, zip(*samples))
        return s_batch, a_batch, r_batch, ns_batch, done_batch

    def get_size(self):
        return len(self.memory)

# ---------------------
# Surrogate Gradient Spike Function
# ---------------------
class SurrGradSpike(torch.autograd.Function):
    scale = 100.0

    @staticmethod
    def forward(ctx, input):
        ctx.save_for_backward(input)
        out = torch.zeros_like(input)
        out[input > 0] = 1.0
        return out

    @staticmethod
    def backward(ctx, grad_output):
        input, = ctx.saved_tensors
        grad_input = grad_output.clone()
        grad = grad_input / (SurrGradSpike.scale * torch.abs(input) + 1.0) ** 2
        return grad

# ---------------------
# DSNN Model
# ---------------------
class DSNN(nn.Module):
    def __init__(self, architecture, seed, alpha, beta, weight_scale, batch_size, 
                 threshold, simulation_time, learning_rate, reset_potential=0):
        super().__init__()
        self.architecture = architecture
        self.simulation_time = simulation_time
        self.batch_size = batch_size
        self.threshold = threshold
        self.reset_potential = reset_potential
        self.alpha = alpha
        self.beta = beta
        
        torch.manual_seed(seed)
        
        self.weights = nn.ParameterList()
        for i in range(len(architecture)-1):
            w = torch.Tensor(architecture[i], architecture[i+1])
            nn.init.normal_(w, mean=0.0, std=weight_scale/np.sqrt(architecture[i]))
            self.weights.append(nn.Parameter(w))
            
        self.spike_fn = SurrGradSpike.apply
        self.optimizer = optim.Adam(self.parameters(), lr=learning_rate)

    def forward(self, x):
        batch_size = x.size(0)
        syn, mem, spk = [], [], []
        
        for l in range(len(self.weights)):
            syn.append(torch.zeros(batch_size, self.weights[l].size(1), device=device))
            mem.append(torch.zeros(batch_size, self.weights[l].size(1), device=device))
            spk.append([])
        
        mem_rec = []
        
        for t in range(self.simulation_time):
            input = x[:, t, :]
            
            for l in range(len(self.weights)):
                if l == 0:
                    h = torch.mm(input, self.weights[l])
                else:
                    h = torch.mm(spk[l-1][-1], self.weights[l])
                
                syn[l] = self.alpha * syn[l] + h
                mem[l] = self.beta * mem[l] + syn[l]
                
                if l < len(self.weights)-1:
                    mthr = mem[l] - self.threshold
                    spk_current = self.spike_fn(mthr)
                    mem[l] = mem[l] * (1 - spk_current) + self.reset_potential * spk_current
                    spk[l].append(spk_current)
                
                if l == len(self.weights)-1:
                    mem_rec.append(mem[l])
        
        q_values = mem[-1]
        return q_values, mem_rec, spk

# ---------------------
# DQN Agent with DSNN
# ---------------------
class DQN(BaseModel):
    def __init__(self, discount_factor: float, epsilon: float, e_min: int, e_max: int, dsnn_config: dict):
        super().__init__(discount_factor, epsilon, e_min, e_max)
        self.gamma = discount_factor
        self.epsilon_greedy = EpsilonGreedy(epsilon)
        self.e_min = e_min
        self.exp_replay = ExperienceReplay(e_max)
        self.batch_size = dsnn_config['batch_size']
        self.simulation_time = dsnn_config['simulation_time']

        self.training_net = DSNN(**dsnn_config).to(device)
        self.target_net = DSNN(**dsnn_config).to(device)
        self.update_target_network()
        
        self.population_size = 3
        self.cache = []

    def population_encode(self, state):
        """Convert board state to population coding with temporal dynamics"""
        encoded = torch.zeros(self.simulation_time, 9 * self.population_size, device=device)
        
        for t in range(self.simulation_time):
            pop_code = []
            for val in state:
                noisy_val = val + np.random.normal(0, 0.05)
                if val == 1:  # X
                    pop_code += [
                        np.exp(-((noisy_val - -2.0)**2) / 0.01),
                        np.exp(-((noisy_val -  0.0)**2) / 0.01),
                        np.exp(-((noisy_val -  1.0)**2) / 0.01)
                    ]
                elif val == -1:  # O
                    pop_code += [
                        np.exp(-((noisy_val - -1.0)**2) / 0.01),
                        np.exp(-((noisy_val -  1.0)**2) / 0.01),
                        np.exp(-((noisy_val -  2.0)**2) / 0.01)
                    ]
                else:  # empty
                    pop_code += [
                        np.exp(-((noisy_val - -2.0)**2) / 0.01),
                        np.exp(-((noisy_val -  0.0)**2) / 0.01),
                        np.exp(-((noisy_val -  2.0)**2) / 0.01)
                    ]
            
            encoded[t] = torch.tensor(pop_code, device=device)
        
        return encoded.unsqueeze(0)

    def observe(self, state, action_space: list = None):
        with torch.no_grad():
            encoded_state = self.population_encode(state)
            q_values, _, _ = self.training_net(encoded_state)
            q_values = q_values.cpu().numpy().flatten()

        if action_space is not None:
            valid_q = [(q_values[a], a) for a in action_space]
            return max(valid_q, key=lambda x: x[0])[1]
        return np.argmax(q_values)

    def observe_on_training(self, state, action_space: list = None) -> int:
        with torch.no_grad():
            encoded_state = self.population_encode(state)
            q_values, _, _ = self.training_net(encoded_state)
            q_values = q_values.cpu().numpy().flatten()

        action = self.epsilon_greedy.perform(q_values, action_space)
        self.cache.extend([state, action])
        return action

    def take_reward(self, reward, next_state, done):
        self.cache.extend([reward, next_state, done])
        self.exp_replay.add_experience(self.cache.copy())
        self.cache.clear()

    def train_network(self, sample_size: int, batch_size: int):
        if self.exp_replay.get_size() < self.e_min:
            return None

        states, actions, rewards, next_states, dones = self.exp_replay.sample_experience(sample_size, cer_mode=False)
        
        state_batch = torch.stack([self.population_encode(s) for s in states]).squeeze(1)
        next_state_batch = torch.stack([self.population_encode(ns) for ns in next_states]).squeeze(1)
        
        action_batch = torch.LongTensor(actions).to(device)
        reward_batch = torch.FloatTensor(rewards).to(device)
        done_batch = torch.BoolTensor(dones).to(device)

        with torch.no_grad():
            next_q_values, _, _ = self.target_net(next_state_batch)
            max_next_q = next_q_values.max(1)[0]
            target_q = reward_batch + (1 - done_batch.float()) * self.gamma * max_next_q

        current_q, mem_rec, spk_rec = self.training_net(state_batch)
        current_q = current_q.gather(1, action_batch.unsqueeze(1)).squeeze(1)

        self.training_net.optimizer.zero_grad()
        loss = F.mse_loss(current_q, target_q)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.training_net.parameters(), max_norm=1.0)
        self.training_net.optimizer.step()

        return loss.item()

    def update_target_network(self):
        self.target_net.load_state_dict(self.training_net.state_dict())

    def save_model(self, filename):
        torch.save({
            'training_net': self.training_net.state_dict(),
            'target_net': self.target_net.state_dict(),
            'epsilon': self.epsilon_greedy.epsilon
        }, filename)

    def load_model(self, filename):
        checkpoint = torch.load(filename)
        self.training_net.load_state_dict(checkpoint['training_net'])
        self.target_net.load_state_dict(checkpoint['target_net'])
        self.epsilon_greedy.epsilon = checkpoint['epsilon']


def analyze_network(agent, state):
    encoded_state = agent.rate_encode(state)
    q_values, mem_rec, spk_rec = agent.training_net(encoded_state)
    
    print("Final Q-values:", q_values.detach().cpu().numpy())
    print("Output layer membrane potentials:", mem_rec[-1].detach().cpu().numpy())
    print("Hidden layer spikes:", [s.sum().item() for s in spk_rec[0]])

# ---------------------
# Training Configuration
# ---------------------
dsnn_config = {
    'architecture': [27, 128, 128, 9],
    'seed': 42,
    'alpha': 0.9,
    'beta': 0.85,
    'weight_scale': 0.15,
    'batch_size': 32,
    'threshold': 0.1,
    'simulation_time': 1,
    'learning_rate': 0.0001,
    'reset_potential': 0.0
}

env = Tictactoe_v0()
agent = DQN(
    discount_factor=0.95,
    epsilon=1.0,
    e_min=1000,
    e_max=100000,
    dsnn_config=dsnn_config
)

agent.update_target_network()

num_episodes = 30001
batch_size = 32
sample_size = 64

total_loss = 0
episode_count_for_avg_loss = 0
win_count = 0
loss_count = 0
draw_count = 0



def print_spikes(spk_rec, timestep=-1):
    """Print spike activity for last timestep"""
    print("\nSpike Activity:")
    for layer_idx, layer_spikes in enumerate(spk_rec):
        if len(layer_spikes) > 0:
            spike_count = layer_spikes[timestep].sum().item()
            print(f"Layer {layer_idx+1}: {spike_count} spikes")

def print_all_spikes(spk_rec):
    for layer_idx, layer_spikes_list in enumerate(spk_rec):
        if not layer_spikes_list:  # empty list?
            print(f"[Debug] Layer {layer_idx+1} has no spike data.")
            continue
        
        print(f"\n[Debug] Layer {layer_idx+1} Spikes:")
        layer_spikes_tensor = torch.stack(layer_spikes_list, dim=0)  # shape => (T, batch_size, layer_size)
        print(layer_spikes_tensor)

            
def print_encoded_state(encoded_state, timesteps=3):
    """Print population encoding for first few timesteps"""
    print("\nPopulation Encoding (All Timesteps):")
    encoded_np = encoded_state.squeeze(0).cpu().numpy()
    
    for t in range(encoded_np.shape[0]):  # Iterate through all timesteps
        print(f"\nTimestep {t+1}:")
        for pos in range(9):
            neurons = encoded_np[t, pos*3 : (pos+1)*3]
            print(f"Pos {pos}: [{neurons[0]:.2f}, {neurons[1]:.2f}, {neurons[2]:.2f}]")

def print_membrane_potentials(q_values, action):
    """Show output layer decision process"""
    print("\nOutput Membrane Potentials (Q-values):")
    q_np = q_values.detach().cpu().numpy().flatten()
    for i in range(9):
        print(f"Position {i}: {q_np[i]:.2f}")
    print(f"Selected Action: Position {action} (Q-value: {q_np[action]:.2f})")

# ---------------------
# Training Loop
# ---------------------
import wandb
wandb.login(key='e21de1f4d4c13b4ba109db92ba20cc946e7da3c5') 
wandb.init(project="thesis", name="DSQN going second against minimax 128")

for episode in range(num_episodes):
    state = env.reset(is_human_first=False) #If True DSQN plays first. 
    # print(f"[DEBUG] AI is playing as {'X' if env.player_mark == 1 else 'O'}")
    # print(f"[DEBUG] AI goes {'first' if env.player_mark == 1 else 'second'}")
    done = False
    episode_reward = 0

    # uncomment the below
    # print(f"\n--- Episode {episode + 1} ---")
    # env.render()
    
    while not done:
        action_space = [i for i, val in enumerate(state) if val == 0]
        action = agent.observe_on_training(state, action_space)
        next_state, reward, done, _ = env.step(action)
        episode_reward += reward

        if episode % 100 == 0:
            encoded_state = agent.population_encode(state)
            #uncomment the below
            # print_encoded_state(encoded_state)
            # env.render()
            with torch.no_grad():
                q_values, mem_rec, spk_rec = agent.training_net(encoded_state)
            
            # Show membrane potentials
            #uncomment the below
            # print_membrane_potentials(q_values[0], action)
            # env.render()

            # Show spike activity
            #uncomment the below
            # print_spikes(spk_rec)
            # print_all_spikes(spk_rec)

        agent.take_reward(reward, next_state, done)

        if agent.exp_replay.get_size() > agent.e_min:
            loss = agent.train_network(sample_size, batch_size)
            if loss is not None:
                total_loss += loss
                episode_count_for_avg_loss += 1
            agent.update_target_network()

        state = next_state

    agent.epsilon_greedy.decay(decay_value=0.995, lower_bound=0.01)

    if episode_reward == 1:
        win_count += 1
    elif episode_reward == -1:
        loss_count += 1
    else:
        if done:
            draw_count += 1

    if episode % 100 == 0:
        avg_loss = total_loss / episode_count_for_avg_loss if episode_count_for_avg_loss > 0 else 0
        combined_rate = win_count + draw_count
        print(f"Episode Summary {episode}, Games Won: {win_count}, Games Lost: {loss_count}, Games won+draw: {combined_rate}, Games Drawn: {draw_count}")
        print(f"Episode: {episode}, Avg Loss: {avg_loss:.4f}")

        wandb.log({
            "Episode": episode,
            "Win Rate": win_count,
            "Loss Rate": loss_count,
            "Draw Rate": draw_count,
            "Win + Draw Rate": combined_rate,
            "Average Loss": avg_loss,
        })

        total_loss = 0.0
        episode_count_for_avg_loss = 0
        win_count = 0
        loss_count = 0
        draw_count = 0

os.makedirs("saved_models", exist_ok=True)
agent.save_model(filename=os.path.join("saved_models", "tictactoe_dsnn_vs_minimax1_step1.pth"))
wandb.finish()